In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

Player value rankings

In [0]:
gold_value = spark.sql("""
    SELECT
        p.player_id,
        p.display_name,
        t.name AS team_name,
        pos.singular_name_short AS position,
        p.current_price,
        p.total_points,
        ROUND(p.total_points / NULLIF(p.current_price, 0), 2) AS points_per_million,
        p.minutes,
        ROUND(p.total_points / NULLIF(p.minutes / 90.0, 0), 2) AS points_per_90,
        p.xG,
        p.xA,
        p.xGI,
        p.ownership_pct,
        p.gw_transfers_in - p.gw_transfers_out AS net_transfers
    FROM fpl_project.silver.players p
    JOIN fpl_project.silver.teams t ON p.team_id = t.team_id
    JOIN fpl_project.silver.positions pos ON p.position_id = pos.id
    WHERE p.minutes > 0
    ORDER BY points_per_million DESC
""")

gold_value.write.mode("overwrite").format("delta") \
    .saveAsTable("fpl_project.gold.player_value_rankings")

print("Gold player value rankings built.")

rolling form

In [0]:
w = Window.partitionBy("player_id").orderBy("gameweek") \
    .rowsBetween(-4, 0)

latest_gw = Window.partitionBy("player_id").orderBy(F.desc("gameweek"))

gold_form = spark.table("fpl_project.silver.player_history") \
    .withColumn("rolling_5gw_points", F.sum("total_points").over(w)) \
    .withColumn("rolling_5gw_minutes", F.sum("minutes").over(w)) \
    .withColumn("rolling_5gw_xGI", F.sum("xG").over(w) + F.sum("xA").over(w)) \
    .withColumn("rolling_5gw_bonus", F.sum("bonus").over(w)) \
    .withColumn("gw_rank", F.row_number().over(latest_gw)) \
    .filter("gw_rank = 1") \
    .drop("gw_rank")

gold_form.write.mode("overwrite").format("delta") \
    .saveAsTable("fpl_project.gold.player_form")

print("Gold player form built.")

Fixture difficulty and differentials

In [0]:
gold_upcoming = spark.sql("""
    SELECT
        t.name AS team_name,
        f.gameweek,
        opp.name AS opponent,
        CASE WHEN f.home_team_id = t.team_id THEN 'H' ELSE 'A' END AS venue,
        CASE WHEN f.home_team_id = t.team_id
             THEN f.home_difficulty ELSE f.away_difficulty END AS difficulty
    FROM fpl_project.silver.fixtures f
    JOIN fpl_project.silver.teams t
        ON t.team_id IN (f.home_team_id, f.away_team_id)
    JOIN fpl_project.silver.teams opp
        ON opp.team_id = CASE
            WHEN f.home_team_id = t.team_id THEN f.away_team_id
            ELSE f.home_team_id END
    WHERE f.finished = false
    ORDER BY t.name, f.gameweek
""")

gold_upcoming.write.mode("overwrite").format("delta") \
    .saveAsTable("fpl_project.gold.fixture_difficulty")

print("Gold fixture difficulty built.")

# Differentials: low ownership, high value
gold_diff = spark.sql("""
    SELECT *
    FROM fpl_project.gold.player_value_rankings
    WHERE ownership_pct < 10.0
      AND total_points > 50
      AND minutes > 500
    ORDER BY points_per_million DESC
""")

gold_diff.write.mode("overwrite").format("delta") \
    .saveAsTable("fpl_project.gold.differentials")

print("Gold differentials built.")